Author: Daniel Abadjiev  
Date: October 27, 2026  
Description: Attempts to explore the edm4hep format and particles inside it.  

In [36]:
import numpy as np
# import pyLCIO
# import ROOT
import uproot
from math import *
import argparse
import os
import random
from plothelper import *
from datetime import datetime
import pandas as pd


file_list = ["./sim_output.edm4hep.root","./sim_output2.edm4hep.root","./sim_output3.edm4hep.root","./digi_real.edm4hep.root"]
filepath = file_list[3]

In [2]:
f = uproot.open(filepath)
print(f.keys())
print(f["events"].keys())
print(f["events"]["VertexBarrelCollection"])
print(f["events"]["VertexBarrelCollection"].keys())
vertexEventsDF = f["events"]["VertexBarrelCollection"].arrays(library="pd")
vertexEventsDF2 = f["events"]["_VertexBarrelCollection_particle"].arrays(library="pd")

outerEventsDF = f["events"]["OuterTrackerBarrelCollection"].arrays(library="pd")
outerEventsDF2 = f["events"]["_OuterTrackerBarrelCollection_particle"].arrays(library="pd")
mcparticles = f["events"]["MCParticles"].arrays(library="pd")
links = f["events"]["VXDBarrelRealHitsRelations"].arrays(library="pd")
linksTo = f["events"]["_VXDBarrelRealHitsRelations_to"].arrays(library="pd")
linksFrom = f["events"]["_VXDBarrelRealHitsRelations_from"].arrays(library="pd")


['configuration_metadata;1', 'events;1', 'metadata;1', 'podio_metadata;1']
['ECalBarrelCollection', 'ECalBarrelCollection/ECalBarrelCollection.cellID', 'ECalBarrelCollection/ECalBarrelCollection.energy', 'ECalBarrelCollection/ECalBarrelCollection.position.x', 'ECalBarrelCollection/ECalBarrelCollection.position.y', 'ECalBarrelCollection/ECalBarrelCollection.position.z', 'ECalBarrelCollection/ECalBarrelCollection.contributions_begin', 'ECalBarrelCollection/ECalBarrelCollection.contributions_end', '_ECalBarrelCollection_contributions', '_ECalBarrelCollection_contributions/_ECalBarrelCollection_contributions.index', '_ECalBarrelCollection_contributions/_ECalBarrelCollection_contributions.collectionID', 'ECalBarrelCollectionContributions', 'ECalBarrelCollectionContributions/ECalBarrelCollectionContributions.PDG', 'ECalBarrelCollectionContributions/ECalBarrelCollectionContributions.energy', 'ECalBarrelCollectionContributions/ECalBarrelCollectionContributions.time', 'ECalBarrelCollectionContr

In [23]:
def collectionDfFromTree(f: uproot.reading.ReadOnlyDirectory,collectionBranch = "VertexBarrelCollection",collectionBranchIndex=""):
    assert isinstance(f,uproot.reading.ReadOnlyDirectory)
    assert "events;1" in f.keys()
    if collectionBranchIndex=="":
        collectionBranchIndex = "_"+collectionBranch+"_particle"
    assert collectionBranch in f["events"].keys()
    assert collectionBranchIndex in f["events"].keys()
    df1 = f["events"][collectionBranch].arrays(library="pd")
    df2 = f["events"][collectionBranchIndex].arrays(library="pd")
    eventsDf = df1.merge(df2,how='outer',left_index=True,right_index=True)
    for idx,row in eventsDf.iterrows():
        numHits = [len(row[key]) for key in row.keys()]
        assert len(set(numHits)) == 1
    return eventsDf



In [ ]:
print(type(f))
assert isinstance(f,uproot.reading.ReadOnlyDirectory)
print(f.keys())

<class 'uproot.reading.ReadOnlyDirectory'>
['configuration_metadata;1', 'events;1', 'metadata;1', 'podio_metadata;1']


In [24]:
# vertexEventsDf = vertexEventsDF.merge(vertexEventsDF2,how='outer',left_index=True,right_index=True)
# # print(vertexEventsDf)
# print(vertexEventsDf.keys())
# for idx,row in vertexEventsDf.iterrows():
#     # print(row)
#     numHits = [len(row[key]) for key in row.keys()]
#     assert len(set(numHits)) == 1
vertexEventsDf = collectionDfFromTree(f)
print(vertexEventsDf)
    

                       VertexBarrelCollection.cellID  \
0           [50364417, 50364545, 67141889, 67141889]   
1  [50454529, 50454657, 50446593, 67273217, 67339...   
2              [16867329, 16859137, 16867457, 82177]   
3       [16785409, 16785537, 16785665, 16897, 25345]   
4  [33611777, 33611905, 50389249, 50405889, 50438...   
5  [50446337, 50446465, 50438401, 50487809, 50479...   
6  [33644545, 33644673, 16859393, 16900609, 16941...   
7                     [50372609, 50372737, 67150081]   
8     [16875521, 16875649, 16867585, 123393, 172801]   
9                                                 []   

                         VertexBarrelCollection.eDep  \
0  [1.454484572604997e-05, 1.8365239156992175e-05...   
1  [1.1806205293396488e-05, 1.4764224033569917e-0...   
2  [1.808064007491339e-05, 2.8722470233333297e-05...   
3  [1.5276906196959317e-05, 1.1402589734643698e-0...   
4  [1.3874917385692243e-05, 1.3680019947059918e-0...   
5  [3.3173459087265655e-05, 1.405437069479376e-

In [41]:
hitRows = []
for idx,row in vertexEventsDf.iterrows():
    indexFromKey = "_VertexBarrelCollection_particle.index"
    for ihit in range(len(row[indexFromKey])):
        hitRow = {".".join(key.split(".")[1:]):row[key][ihit] for key in row.keys()}
        # for key in row.keys():
        #     print(".".join(key.split(".")[1:]),row[key][ihit])
        hitRows.append(hitRow)

In [38]:
hitDf = pd.DataFrame(hitRows)
print(hitDf)

      cellID      eDep      time  pathLength     quality  position.x  \
0   50364417  0.000015  0.139147    0.069076           0    0.166005   
1   50364545  0.000018  0.148595    0.069078           0    0.164893   
2   67141889  0.000017  0.237301    0.069423           0    0.076187   
3   67141889  0.000071  0.237606    0.264609  1073741824   -0.204008   
4   50454529  0.000012  0.115823    0.057527           0   28.729155   
5   50454657  0.000015  0.123690    0.057529           0   30.684081   
6   50446593  0.000012  0.196868    0.057683           0   48.879789   
7   67273217  0.000021  0.283859    0.057344           0   70.536461   
8   67339009  0.000015  0.392731    0.057547           0   97.678837   
9   67330817  0.000014  0.393855    0.057847           0   97.959367   
10  16867329  0.000018  0.150477    0.074697           0  -15.072208   
11  16859137  0.000029  0.154760    0.076879           0  -15.500149   
12  16867457  0.000021  0.160692    0.074692           0  -16.09

In [48]:
hitDf.filter(cond=hitDf["index"]==0)

,cellID,eDep,time,pathLength,quality,position.x,position.y,position.z,momentum.x,momentum.y,momentum.z,index,collectionID
0,50364417,0.000015,0.139147,0.069076,0,0.166005,30.165000,28.752296,-0.000333,4.079331,3.888560,0,2714477136
1,50364545,0.000018,0.148595,0.069078,0,0.164893,32.215000,30.706480,-0.003657,4.079118,3.888759,0,2714477136
2,67141889,0.000017,0.237301,0.069423,0,0.076187,51.454839,49.057587,-0.033081,4.076814,3.889820,0,2714477136
3,67141889,0.000071,0.237606,0.264609,1073741824,-0.204008,51.400251,49.047387,-0.000644,-0.000029,0.000198,0,2714477136
4,50454529,0.000012,0.115823,0.057527,0,28.729155,-9.466629,16.986593,7.600472,-2.480031,4.488900,0,2714477136
5,50454657,0.000015,0.123690,0.057529,0,30.684081,-10.103927,18.141061,7.601489,-2.476961,4.488846,0,2714477136
6,50446593,0.000012,0.196868,0.057683,0,48.879789,-16.008125,28.877551,7.609118,-2.454302,4.488136,0,2714477136
7,67273217,0.000021,0.283859,0.057344,0,70.536461,-22.942009,41.643377,7.618674,-2.422507,4.489060,0,2714477136
8,67339009,0.000015,0.392731,0.057547,0,97.678837,-31.491957,57.622644,7.631687,-2.381623,4.488656,0,2714477136
9,67330817,0.000014,0.393855,0.057847,0,97.959367,-31.578543,57.787481,7.631680,-2.381343,4.488688,0,2714477136


In [11]:
print([(mcparticles["MCParticles.PDG"][i])for i in range(10)])

[<Array [13, 11] type='2 * int32'>, <Array [13] type='1 * int32'>, <Array [13] type='1 * int32'>, <Array [13] type='1 * int32'>, <Array [13, 11] type='2 * int32'>, <Array [13] type='1 * int32'>, <Array [13] type='1 * int32'>, <Array [13] type='1 * int32'>, <Array [13] type='1 * int32'>, <Array [13, 11] type='2 * int32'>]


In [13]:
mcparticles.iloc[0]

MCParticles.PDG                                                     [13.0 11.0]
MCParticles.generatorStatus                                           [1.0 0.0]
MCParticles.simulatorStatus                           [33554432.0 1493172224.0]
MCParticles.charge                                                  [-1.0 -1.0]
MCParticles.time                                       [0.0 1.5657464265823364]
MCParticles.mass                                       [0.105658 0.00051099891]
MCParticles.vertex.x                                  [0.0 -17.722444167138782]
MCParticles.vertex.y                                  [-0.0 338.92905348336257]
MCParticles.vertex.z                                   [0.0 323.91187798494366]
MCParticles.endpoint.x                [-14978.325317703111 -17.906132151229126]
MCParticles.endpoint.y                    [26680.43948317624 340.1464476137431]
MCParticles.endpoint.z                              [30000.0 324.5228534862669]
MCParticles.momentum.x              [0.0

In [16]:
vertexEventsDF.iloc[0]

VertexBarrelCollection.cellID            [50364417.0 50364545.0 67141889.0 67141889.0]
VertexBarrelCollection.eDep          [1.454484572604997e-05 1.8365239156992175e-05 ...
VertexBarrelCollection.time          [0.1391465961933136 0.14859545230865479 0.2373...
VertexBarrelCollection.pathLength    [0.06907620280981064 0.06907816231250763 0.069...
VertexBarrelCollection.quality                              [0.0 0.0 0.0 1073741824.0]
VertexBarrelCollection.position.x    [0.16600517596864428 0.16489298773069 0.076186...
VertexBarrelCollection.position.y    [30.165 32.214999999999996 51.45483864807524 5...
VertexBarrelCollection.position.z    [28.75229635761209 30.706479803299622 49.05758...
VertexBarrelCollection.momentum.x    [-0.000332693918608129 -0.003656902117654681 -...
VertexBarrelCollection.momentum.y    [4.079331398010254 4.079117774963379 4.0768136...
VertexBarrelCollection.momentum.z    [3.8885602951049805 3.888758659362793 3.889819...
Name: 0, dtype: awkward

In [15]:
outerEventsDF2.iloc[0]

_OuterTrackerBarrelCollection_particle.index                             [0 0]
_OuterTrackerBarrelCollection_particle.collectionID    [2714477136 2714477136]
Name: 0, dtype: awkward

In [7]:
for b in ["momentum", "momentumAtEndpoint", "endpoint"]:
    print(b, [mcparticles[f"MCParticles.{b}.{c}"][8][0] for c in "xyz"])

momentum [np.float64(-0.4620397954065685), np.float64(-4.559888805327136), np.float64(-2.400232964602328)]
momentumAtEndpoint [np.float64(-0.0), np.float64(-0.0), np.float64(-0.0)]
endpoint [np.float64(-152.6292080241181), np.float64(-1506.3036213638443), np.float64(-792.8876692066572)]
